# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya Exploration with `mlcroissant`
This notebook demonstrates how to load, explore, and process the FAIR^2 dataset using the `mlcroissant` library. All data elements—record sets, fields, columns—are referenced by their `@id` for full transparency and reproducibility.

### Dataset Source
The dataset is defined by a Croissant schema available at:

https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json

In [ ]:
# Ensure mlcroissant is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import matplotlib.pyplot as plt

# Set dataset URL (Croissant schema)
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load dataset metadata
dataset = mlc.Dataset(croissant_url)

# Access metadata properties
metadata = dataset.metadata

print(f"Dataset Name: {metadata.name}\n")
print(f"Description: {metadata.description}\n")
print(f"Identifier: {metadata.identifier}\n")
print(f"License: {metadata.license}\n")

## 2. Data Overview
Review available record sets, fields, and their `@id`s in the dataset.

We enumerate all record sets using their `@id`. For each record set, we list available fields and their respective `@id`s.

In [ ]:
# List all record sets by @id
record_sets = dataset.record_sets()
print("Record sets available in dataset:")
for rs in record_sets:
    print(f"  - RecordSet @id: {rs['@id']} | Name: {rs.get('name', 'N/A')}")
    # List fields inside this record set
    fields = rs.get('field', [])
    if fields:
        print(f"    Fields (@id):")
        for fld in fields:
            if isinstance(fld, dict):
                print(f"      - {fld.get('@id')} (label: {fld.get('name', 'N/A')})")
            else:
                print(f"      - {fld}")
    else:
        print("    No fields listed.")
print()
# Optionally, print the first few records from each record set, referenced by @id
for rs in record_sets:
    rs_id = rs['@id']
    print(f"First two records from RecordSet {rs_id}:")
    try:
        for i, rec in enumerate(dataset.records(record_set=rs_id)):
            if i < 2:
                print(rec)
            else:
                break
    except Exception as e:
        print(f"  ERROR loading records: {e}")
    print("-")

## 3. Data Extraction
Load data from each record set into a DataFrame for analysis.

Refer to the specific `@id` values of the record sets and fields from the overview above.

In [ ]:
# Extract all data from record sets using @id
record_set_ids = [rs['@id'] for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded DataFrame for record set {rs_id}. Columns:")
        print(dataframes[rs_id].columns.tolist())
    else:
        print(f"No records found for {rs_id}.")
    print()

# As an example, select the first record set for further analysis
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id and main_record_set_id in dataframes:
    print(f"Preview records from {main_record_set_id}:")
    display(dataframes[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps:
- Filtering records based on a numeric field
- Normalizing a field
- Grouping records by a categorical field

All references are by `@id`.

In [ ]:
# Example: Select a numeric field for analysis

if main_record_set_id and main_record_set_id in dataframes:
    df = dataframes[main_record_set_id]
    print(f"Available columns in {main_record_set_id}:")
    print(df.columns.tolist())

    # Identify plausible numeric field and categorical group field by @id
    # Example: suppose the field @id for log likelhood is 'http://senscience.ai/logLikelihood' and group by 'http://senscience.ai/ward'
    numeric_field_id = None
    group_field_id = None
    for col in df.columns:
        if 'loglikelihood' in col.lower():
            numeric_field_id = col
        if 'ward' in col.lower():
            group_field_id = col

    # Use fallback if not found
    numeric_field_id = numeric_field_id or (df.columns[0] if len(df.columns) > 1 else None)
    group_field_id = group_field_id or (df.columns[1] if len(df.columns) > 1 else None)

    if numeric_field_id:
        print(f"Selected numeric field: {numeric_field_id}")
        # Filter records
        threshold = 10
        try:
            # Ensure numeric
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
            filtered_df = df[df[numeric_field_id] > threshold]
            print(f"Filtered records with {numeric_field_id} > {threshold}:")
            display(filtered_df.head())

            # Normalization
            filtered_df[f"{numeric_field_id}_normalized"] = (
                filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
            ) / filtered_df[numeric_field_id].std()
            print(f"Normalized {numeric_field_id} for filtered records:")
            display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

            # Grouping by category
            if group_field_id and group_field_id in df.columns:
                grouped_df = filtered_df.groupby(group_field_id).mean()
                print(f"Grouped and averaged data by {group_field_id}:")
                display(grouped_df.head())
        except Exception as e:
            print(f"Error in numeric EDA: {e}")
    else:
        print("No numeric field available for EDA.")

## 5. Visualization
Visualize distributions and relationships.

*All fields and groupings are referenced by their `@id`s as above.*

In [ ]:
# Plotting numeric field histogram, grouped by category

if main_record_set_id and main_record_set_id in dataframes and numeric_field_id:
    df = dataframes[main_record_set_id]
    # Ensure numeric
    df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
    plt.figure(figsize=(8,4))
    plt.hist(df[numeric_field_id].dropna(), bins=20, color='skyblue', edgecolor='black')
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.ylabel("Count")
    plt.show()

    if group_field_id and group_field_id in df.columns:
        plt.figure(figsize=(8,4))
        df.groupby(group_field_id)[numeric_field_id].mean().plot(kind='bar')
        plt.title(f"Mean {numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(f"Mean {numeric_field_id}")
        plt.show()

## 6. Conclusion
This notebook provides a transparent, reproducible workflow for exploring the FAIR^2 dataset via Croissant and `mlcroissant`.

- All data references use explicit `@id` notation: record sets, fields, categorical and numeric variables.
- You can extend this notebook by adjusting the field and record set selections or by implementing additional transformations and visualizations based on your research needs.

### Key findings
- The dataset provides detailed ordered logistic regression results and socio-demographic predictors in rangeland management.
- Distributions and averages can be examined by ward or other categorical variables.
- All processing steps can be traced using the Croissant schema's globally unique identifiers.